In [1]:
import numpy as np
import pandas as pd
from keras.models import Model
from keras.layers import Input, Dense, concatenate
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib
import pickle
from keras.models import Sequential  # Import the Sequential model
from keras.layers import Dense  # Import Dense layers
from keras_tuner import RandomSearch  # Import RandomSearch tuner
from sklearn.metrics import mean_squared_error
import numpy as np
from keras.models import load_model

# Sample schema for issues
data = {
    "description": ["Failure in motor", "Pump leakage", "Equipment overheating", "Sensor malfunction", "Electrical short"],
    "severity": [8, 5, 7, 4, 9],
    "total_downtime": [120, 45, 80, 30, 90],
    "oee": [0.8, 0.9, 0.7, 0.85, 0.6],
    "failure_modes": ["Motor failure", "Leakage", "Overheating", "Sensor issue", "Short circuit"],
    "timeframe_to_fix": [3, 1, 2, 1, 3]  # Timeframe to predict (hours to fix)
}

# Convert to DataFrame
df = pd.DataFrame(data)

# Text preprocessing with TF-IDF
tfidf = TfidfVectorizer(max_features=11)  # Adjust max_features to match the output shape (11 in this case)
X_text = tfidf.fit_transform(df['description'].values).toarray()  # This will give shape (n_samples, max_features)

# Numeric features (severity, downtime, OEE)
numeric_features = df[['severity', 'total_downtime', 'oee']].values
scaler = StandardScaler()
numeric_features_scaled = scaler.fit_transform(numeric_features)

# Targets (timeframe to fix)
timeframe = df['timeframe_to_fix'].values

# Split dataset into training and testing sets
X_train_text, X_test_text, X_train_num, X_test_num, y_train, y_test = train_test_split(
    X_text, numeric_features_scaled, timeframe, test_size=0.2, random_state=42)

# Build the model
# Inputs for text and numeric data
text_input = Input(shape=(X_train_text.shape[1],), name='text_input')  # Matches the shape of TF-IDF vector (e.g., 11)
numeric_input = Input(shape=(3,), name='numeric_input')

# Dense layers for text data
dense_text = Dense(128, activation='relu')(text_input)

# Concatenate text and numeric features
concat_layer = concatenate([dense_text, numeric_input])

# Dense layers after concatenation
dense_1 = Dense(128, activation='relu')(concat_layer)
dense_2 = Dense(64, activation='relu')(dense_1)

# Output layer (predicting timeframe)
output = Dense(1, activation='linear', name='timeframe_output')(dense_2)

# Define and compile the model
model = Model(inputs=[text_input, numeric_input], outputs=output)
model.compile(optimizer='adam', loss='mean_squared_error')

# Train the model
model.fit([X_train_text, X_train_num], y_train, epochs=10, batch_size=32, validation_data=([X_test_text, X_test_num], y_test))

# Function to analyze and recommend solutions
def recommend_solution(description, tfidf_model, numeric_data, issue_frequency):
    # Convert the new description to TF-IDF format
    description_vec = tfidf_model.transform([description]).toarray()

    # Predict timeframe using the trained model
    predicted_time = model.predict([description_vec, numeric_data])

    # Simple recommendation logic (can be extended)
    if issue_frequency > 5:
        recommended_solution = "This issue occurs frequently. Consider preventive maintenance or upgrading equipment."
    else:
        recommended_solution = "The issue is rare. Proceed with standard troubleshooting procedures."

    # Weighted time based on frequency
    frequency_weight = 1 + (issue_frequency / 10)
    weighted_time = predicted_time[0][0] * frequency_weight

    return recommended_solution, weighted_time, frequency_weight

# Test the Recommendation System
def test_model(issue_desc, issue_frequency, numeric_test_values):
    recommended_solution, resolution_time, frequency_weight = recommend_solution(issue_desc, tfidf, numeric_test_values, issue_frequency)
    
    # Output the results
    print(f"Issue Description: {issue_desc}")
    print(f"Recommended Solution: {recommended_solution}")
    print(f"Predicted Resolution Time: {resolution_time:.2f} hours")
    print(f"Frequency Weight Applied: {frequency_weight:.2f}")

# Example input and test
issue_description = "Equipment overheating"
issue_frequency = 6  # Issue occurred 6 times recently
numeric_test_values = np.array([[7, 80, 0.7]])  # Example numeric values for severity, total_downtime, and oee

test_model(issue_description, issue_frequency, numeric_test_values) 

# Save model using the new Keras format
model.save('issue_predictor_model.keras')

# Load the saved model
# Load the model from the saved file
loaded_model = load_model('issue_predictor_model.keras')

def recommend_solution(description, tfidf_model, numeric_data, issue_frequency):
    # Convert the new description to TF-IDF format
    description_vec = tfidf_model.transform([description]).toarray()

    # Predict timeframe using the trained model
    predicted_time = loaded_model.predict([description_vec, numeric_data])

    # Simple recommendation logic (can be extended)
    if issue_frequency > 5:
        recommended_solution = "This issue occurs frequently. Consider preventive maintenance or upgrading equipment."
    else:
        recommended_solution = "The issue is rare. Proceed with standard troubleshooting procedures."

    # Weighted time based on frequency
    frequency_weight = 1 + (issue_frequency / 10)
    weighted_time = predicted_time[0][0] * frequency_weight

    return recommended_solution, weighted_time

# Test function to include recommendation
def test_model(issue_desc, severity, downtime, oee, issue_frequency):
    # Convert the issue description to TF-IDF format
    issue_vec = tfidf.transform([issue_desc]).toarray()
    
    # Scale the numeric features
    numeric_features = np.array([[severity, downtime, oee]])
    numeric_features_scaled = scaler.transform(numeric_features)
    
    # Get the recommended solution and weighted time
    recommended_solution, predicted_timeframe = recommend_solution(issue_desc, tfidf, numeric_features_scaled, issue_frequency)
    
    # Print out the results
    print(f"Issue Description: {issue_desc}")
    print(f"Recommended Solution: {recommended_solution}")
    print(f"Predicted Time to Fix: {predicted_timeframe:.2f} hours")

#test case
test_model("Pump malfunction", 7, 85, 0.75, issue_frequency=6)

# Predict on the test set
y_pred = model.predict([X_test_text, X_test_num])

# Calculate MSE and RMSE
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
print(f'RMSE: {rmse}')


# Define the model-building function
def build_model(hp):
    model = Sequential()
    model.add(Dense(units=hp.Int('units', min_value=32, max_value=512, step=32), activation='relu', input_shape=(11,)))
    model.add(Dense(1, activation='linear'))
    
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

# Instantiate the tuner
tuner = RandomSearch(build_model, objective='val_loss', max_trials=5)

# Start hyperparameter tuning
tuner.search([X_train_text, X_train_num], y_train, epochs=20, validation_data=([X_test_text, X_test_num], y_test))

# Save scaler
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save tfidf model
with open('tfidf.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

#numeric data (replace with your training data)
numeric_data = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]])

# Initialize the scaler
scaler = StandardScaler()

# Fit the scaler on your numeric features
scaler.fit(numeric_data)

# Save the scaler to a file
joblib.dump(scaler, 'scaler.joblib')

print("Scaler saved successfully!")

# Assuming 'tfidf' is your fitted TF-IDF model
joblib.dump(tfidf, 'tfidf_vectorizer.joblib')

from joblib import dump

# Assuming 'model' is your trained model
dump(model, 'maintenance_issue_model.joblib')


Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 12s 12s/step - loss: 5.6776 - val_loss: 0.7657
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 173ms/step - loss: 5.1502 - val_loss: 0.6362
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - loss: 4.6461 - val_loss: 0.5233
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - loss: 4.2085 - val_loss: 0.4225
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - loss: 3.8222 - val_loss: 0.3356
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - loss: 3.4641 - val_loss: 0.2567
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - loss: 3.1249 - val_loss: 0.1906
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - loss: 2.7902 - val_loss: 0.1359
Epoch 9/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - loss: 2.4702 - val_loss: 0.0908
Epoch 10/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - loss: 2.1628 - val_loss: 0.0554
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
Issue Description: Equipment overheating
Recommended Solution: This issue occurs frequently. Consider preventive maintenance or up

['maintenance_issue_model.joblib']